
# STEP 33 — Publication Figure Generator

Notebook นี้สร้างรูปสำหรับบทความ **BrainFMOps-Analyze** จำนวน 5 รูปจากผลการทดลองจริง:

1. `Paper_Fig01_ConfusionMatrix`
2. `Paper_Fig02_ROC`
3. `Paper_Fig03_ThresholdAnalysis`
4. `Paper_Fig04_PrecisionRecall`
5. `Paper_Fig05_Calibration`

คุณสมบัติ:
- ค้นหาไฟล์ผลลัพธ์อัตโนมัติ
- ตรวจจับคอลัมน์ ground truth และ probability อัตโนมัติ
- ใช้ operating threshold = `0.32`
- ส่งออก PNG และ TIFF ที่ 600 dpi
- สร้าง metrics, captions และ manifest อัตโนมัติ

**วิธีใช้:** เปิด Notebook แล้วเลือก `Kernel → Restart & Run All`


In [ ]:

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")

ROOT_CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd(),
]
ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), Path.cwd())

INPUT_CANDIDATES = [
    ROOT / "31B_GroundTruth_Integration" / "evaluation_summary_with_labels.csv",
    ROOT / "evaluation_summary_with_labels.csv",
    ROOT / "evaluation_summary.csv",
]
INPUT_FILE = next((p for p in INPUT_CANDIDATES if p.exists()), None)

OUTPUT_DIR = ROOT / "33_Publication_Figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPERATING_THRESHOLD = 0.32
N_CALIBRATION_BINS = 10
DPI = 600

print("ROOT       :", ROOT)
print("INPUT FILE :", INPUT_FILE)
print("OUTPUT DIR :", OUTPUT_DIR)


In [ ]:

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": DPI,
    "axes.linewidth": 1.0,
    "lines.linewidth": 2.5,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
})
FIGSIZE = (6.5, 5.0)
print("Publication style configured.")


In [ ]:

if INPUT_FILE is None:
    raise FileNotFoundError(
        "ไม่พบ evaluation_summary_with_labels.csv ในโครงการ"
    )

df = pd.read_csv(INPUT_FILE)
print("Shape:", df.shape)
print("Columns:")
for c in df.columns:
    print(" -", c)
df.head()


In [ ]:

def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

TRUE_CANDIDATES = [
    "ground_truth_derived", "ground_truth", "true_label", "label",
    "diagnosis", "class", "target", "y_true"
]
PROB_CANDIDATES = [
    "subject_probability", "subject_level_probability", "ad_probability",
    "predicted_ad_probability", "mean_probability", "probability",
    "prediction_probability", "probability_positive", "positive_probability", "y_prob", "score"
]

true_col = first_existing(df.columns, TRUE_CANDIDATES)
prob_col = first_existing(df.columns, PROB_CANDIDATES)

if true_col is None:
    raise KeyError(f"ไม่พบ ground truth column: {list(df.columns)}")
if prob_col is None:
    raise KeyError(f"ไม่พบ probability column: {list(df.columns)}")

print("Ground-truth column:", true_col)
print("Probability column :", prob_col)


In [ ]:

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()
    positives = {"1","AD","ALZHEIMER","ALZHEIMER'S DISEASE","DEMENTED","POSITIVE","TRUE"}
    negatives = {"0","CN","CONTROL","COGNITIVELY NORMAL","NONDEMENTED","NEGATIVE","FALSE","NORMAL"}
    if text in positives:
        return 1
    if text in negatives:
        return 0
    try:
        num = float(text)
        if num == 1: return 1
        if num == 0: return 0
    except ValueError:
        pass
    return np.nan

work = df.copy()
work["_y_true"] = work[true_col].map(normalize_binary_label)
work["_y_prob"] = pd.to_numeric(work[prob_col], errors="coerce")
work = work[
    work["_y_true"].notna()
    & work["_y_prob"].notna()
    & work["_y_prob"].between(0,1)
].copy()

work["_y_true"] = work["_y_true"].astype(int)
work["_y_pred"] = (work["_y_prob"] >= OPERATING_THRESHOLD).astype(int)

if len(work) == 0:
    raise ValueError("ไม่มีข้อมูลที่ใช้สร้างรูปได้")

print("Usable labelled subjects:", len(work))
print(work["_y_true"].value_counts().sort_index().rename(index={0:"CN",1:"AD"}))


In [ ]:

y_true = work["_y_true"].to_numpy()
y_prob = work["_y_prob"].to_numpy()
y_pred = work["_y_pred"].to_numpy()

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

metrics = {
    "n_subjects": int(len(y_true)),
    "threshold": float(OPERATING_THRESHOLD),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "sensitivity": float(recall_score(y_true, y_pred, zero_division=0)),
    "specificity": float(tn/(tn+fp)) if (tn+fp) else float("nan"),
    "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "average_precision": float(average_precision_score(y_true, y_prob)),
    "brier_score": float(brier_score_loss(y_true, y_prob)),
    "prevalence": float(y_true.mean()),
}
pd.DataFrame([metrics]).T.rename(columns={0:"Value"})


In [ ]:

def save_publication_figure(fig, stem):
    png = OUTPUT_DIR / f"{stem}.png"
    tiff = OUTPUT_DIR / f"{stem}.tiff"
    fig.savefig(png, dpi=DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(tiff, dpi=DPI, bbox_inches="tight", facecolor="white",
                pil_kwargs={"compression":"tiff_lzw"})
    print("Saved:", png.name)
    print("Saved:", tiff.name)


In [ ]:

cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.imshow(cm, cmap="Greys")
ax.set_title("Subject-Level Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Ground-truth class")
ax.set_xticks([0,1], labels=["CN","AD"])
ax.set_yticks([0,1], labels=["CN","AD"])

cutoff = cm.max()/2
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center",
                fontsize=13, color="white" if cm[i,j] > cutoff else "black")

for edge in np.arange(-0.5, 2, 1):
    ax.axhline(edge, linewidth=1, color="black")
    ax.axvline(edge, linewidth=1, color="black")

ax.grid(False)
fig.tight_layout()
save_publication_figure(fig, "Paper_Fig01_ConfusionMatrix")
plt.show()


In [ ]:

fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1], linestyle="--", linewidth=2, label="No-skill reference")
ax.set_title("Receiver Operating Characteristic Curve")
ax.set_xlabel("False-positive rate")
ax.set_ylabel("True-positive rate")
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.grid(True); ax.legend(loc="lower right")
fig.tight_layout()
save_publication_figure(fig, "Paper_Fig02_ROC")
plt.show()


In [ ]:

thresholds = np.linspace(0,1,201)
rows = []
for t in thresholds:
    pred = (y_prob >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    sens = tp_t/(tp_t+fn_t) if (tp_t+fn_t) else np.nan
    spec = tn_t/(tn_t+fp_t) if (tn_t+fp_t) else np.nan
    rows.append({
        "threshold":t,
        "accuracy":accuracy_score(y_true,pred),
        "sensitivity":sens,
        "specificity":spec,
        "f1_score":f1_score(y_true,pred,zero_division=0),
        "balanced_accuracy":np.nanmean([sens,spec]),
    })

threshold_df = pd.DataFrame(rows)
youden_idx = (threshold_df["sensitivity"]+threshold_df["specificity"]-1).idxmax()
youden_t = float(threshold_df.loc[youden_idx,"threshold"])

fig, ax = plt.subplots(figsize=(7.0,5.2))
for col,label in [
    ("accuracy","Accuracy"),
    ("sensitivity","Sensitivity"),
    ("specificity","Specificity"),
    ("f1_score","F1-score"),
    ("balanced_accuracy","Balanced accuracy"),
]:
    ax.plot(threshold_df["threshold"], threshold_df[col], label=label)

ax.axvline(OPERATING_THRESHOLD, linestyle="--", linewidth=2,
           label=f"Predefined threshold = {OPERATING_THRESHOLD:.2f}")
ax.axvline(youden_t, linestyle=":", linewidth=2,
           label=f"Post hoc Youden point = {youden_t:.2f}")

ax.set_title("Subject-Level Performance Across Decision Thresholds")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Metric value")
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.grid(True); ax.legend(loc="best", ncol=2)
fig.tight_layout()

save_publication_figure(fig, "Paper_Fig03_ThresholdAnalysis")
threshold_df.to_csv(OUTPUT_DIR/"Paper_Fig03_ThresholdAnalysis_Data.csv", index=False)
plt.show()


In [ ]:

precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
ap = average_precision_score(y_true, y_prob)
prevalence = y_true.mean()

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(recall_vals, precision_vals, label=f"Average Precision = {ap:.3f}")
ax.axhline(prevalence, linestyle="--", linewidth=2,
           label=f"Positive prevalence = {prevalence:.3f}")
ax.set_title("Subject-Level Precision–Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
ax.grid(True); ax.legend(loc="upper right")
fig.tight_layout()
save_publication_figure(fig, "Paper_Fig04_PrecisionRecall")
plt.show()


In [ ]:

prob_true, prob_pred = calibration_curve(
    y_true, y_prob, n_bins=N_CALIBRATION_BINS, strategy="quantile"
)
brier = brier_score_loss(y_true, y_prob)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(prob_pred, prob_true, marker="o", markersize=6,
        label=f"Observed calibration (Brier score = {brier:.3f})")
ax.plot([0,1],[0,1], linestyle="--", linewidth=2, label="Perfect calibration")
ax.set_title("Subject-Level Calibration Curve")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed AD proportion")
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.grid(True); ax.legend(loc="upper left")
fig.tight_layout()
save_publication_figure(fig, "Paper_Fig05_Calibration")
plt.show()


In [ ]:

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR/"Paper_Subject_Level_Metrics.csv", index=False
)

captions = {
    "Fig. 1": "Subject-level confusion matrix generated by the BrainFMOps-Analyze pipeline at the predefined operating threshold.",
    "Fig. 2": "Receiver operating characteristic curve for subject-level Alzheimer's disease classification.",
    "Fig. 3": "Variation of subject-level classification performance across decision thresholds. The predefined threshold and the post hoc Youden operating point are shown for comparison.",
    "Fig. 4": "Precision–Recall curve for subject-level Alzheimer's disease classification. The dashed horizontal line indicates the positive-class prevalence in the evaluation cohort.",
    "Fig. 5": "Calibration curve comparing predicted Alzheimer's disease probabilities with observed outcome frequencies."
}

manifest = {
    "input_file": str(INPUT_FILE),
    "output_directory": str(OUTPUT_DIR),
    "operating_threshold": OPERATING_THRESHOLD,
    "dpi": DPI,
    "n_labelled_subjects": int(len(work)),
    "metrics": metrics,
    "captions": captions,
}

with open(OUTPUT_DIR/"Paper_Figure_Manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2)

with open(OUTPUT_DIR/"Paper_Figure_Captions.txt","w",encoding="utf-8") as f:
    for k,v in captions.items():
        f.write(f"{k}. {v}\n\n")

print("="*72)
print("PUBLICATION FIGURE PACKAGE CREATED")
print("OUTPUT:", OUTPUT_DIR)
print("="*72)
pd.DataFrame([metrics]).T.rename(columns={0:"Value"})



## ผลลัพธ์ที่ควรได้

โฟลเดอร์ผลลัพธ์:

```text
<repository-root>\33_Publication_Figures
```

ค่าที่ควรตรวจสอบ:

- TN = 36
- FP = 88
- FN = 15
- TP = 73
- ROC-AUC ≈ 0.588
- Average Precision ≈ 0.494
- Brier score ≈ 0.243

หากตัวเลขต่างจากนี้ ให้หยุดก่อนนำรูปไปใส่บทความ เพราะ Notebook อาจเลือกคอลัมน์ผิดชุด
